# Toy BNN results

Pick a dataset below. Shows predictive fit (full batch / minibatch / neg-MAP) and a metrics table.

In [ ]:
import os
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "notebooks") 

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from utils.toy_results_utils import *

DATASET = "gap" #multiscale, hernandez, gap, sharp
SPLIT_ID = 0

# Mode-crossing check (below): reconstruct the full continuous PDMP
# path from saved skeletons (--save-skeleton), or fall back to the
# resampled i.i.d. draws if skeletons were not saved for this run.
USE_SKELETONS = True


In [ ]:
data = load_dataset(DATASET)
any_p = any_payload(DATASET, SPLIT_ID)
bm = make_bm(DATASET, data)

x_test, y_true = data["x_test_raw"], data["y_test_clean_raw"]
x_train, y_train = data["x_train_raw"], data["y_train_raw"]
x_grid = np.sort(x_train if DATASET == "hernandez" else x_test)
si = np.argsort(x_test)

sampler_keys = [s for s in SAMPLER_STYLE
                if any(load_run(r, DATASET, SPLIT_ID, s, skeletons=True) is not None for r in RUN_STYLE)]


In [ ]:
sampler_keys

In [ ]:
fig, axes = plt.subplots(1, len(sampler_keys), figsize=(3.8 * len(sampler_keys), 3.4), sharey=True, squeeze=False)

for col, key in enumerate(sampler_keys):
    ax = axes[0, col]
    base_color, label = SAMPLER_STYLE[key]
    ax.plot(x_test[si], y_true[si], color="0.5", lw=1.3, zorder=3)
    ax.scatter(x_train, y_train, marker="x", color="black", s=35, zorder=5)

    for run_key in RUN_STYLE:
        payload = load_run(run_key, DATASET, SPLIT_ID, key)
        if payload is None:
            continue
        rlabel, ls, lw = RUN_STYLE[run_key]
        color = base_color if run_key == "fullbatch" else RUN_COLOR[run_key]
        mean, epi, total = predictive_summary(payload["samples"], bm, payload, data, x_grid)
        ax.fill_between(x_grid, mean - 2 * total, mean + 2 * total, color=color, alpha=0.15, linewidth=0)
        ax.plot(x_grid, mean, color=color, ls=ls, lw=lw, zorder=4, label=rlabel)

    ax.set_title(label, fontsize=10)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

axes[0, 0].legend(fontsize=8, frameon=False)
fig.suptitle(DATASET, fontsize=11)
fig.tight_layout()
plt.show()


## Metrics

In [ ]:
rows = []
for sampler in GRID_SAMPLERS:
    row = {"sampler": SAMPLER_STYLE.get(sampler, (None, sampler))[1]}
    for run_key in RUN_STYLE:
        payload = load_run(run_key, DATASET, SPLIT_ID, sampler)
        if payload is None:
            continue
        m = predictive_metrics(payload, bm, data)
        row.update({f"{k} ({run_key})": v for k, v in m.items()})
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index("sampler")
cols = [f"{m} ({r})" for m in ("RMSE", "NLL", "CRPS") for r in RUN_STYLE if f"{m} ({r})" in metrics_df.columns]
metrics_df[cols].style.format(precision=3, na_rep="n/a")


## Neg-MAP mode check

Mirror projection: +1 = positive mode, -1 = mirror mode.

In [ ]:
if "negmap" in runs_present():
    mask = flip_mask(bm)
    x_ref_pos = reference_x_ref(DATASET, SPLIT_ID, bm, data)
    rows = []
    for sampler in GRID_SAMPLERS:
        for run_key in RUN_STYLE:
            payload = load_run(run_key, DATASET, SPLIT_ID, sampler)
            if payload is None:
                continue
            proj = mirror_projection(payload["samples"], mask, x_ref_pos)
            rows.append({
                "sampler": SAMPLER_STYLE.get(sampler, (None, sampler))[1],
                "run": RUN_STYLE[run_key][0],
                "proj mean": float(proj.mean()),
                "frac > 0": float((proj > 0).float().mean()),
            })
    proj_df = pd.DataFrame(rows).set_index(["sampler", "run"])
    display(proj_df.style.format(precision=2).background_gradient(subset=["proj mean"], cmap="coolwarm", vmin=-1, vmax=1))
else:
    print("no negmap results")


In [ ]:

mask = flip_mask(bm)
x_ref_pos = reference_x_ref(DATASET, SPLIT_ID, bm, data)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True, sharey=True)
for sampler in ['grid_zigzag', 'grid_sticky_zigzag']:
    payload_fullbatch = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
    payload_minibatch = load_run("minibatch", DATASET, SPLIT_ID, sampler)
    payload_negmap = load_run("negmap", DATASET, SPLIT_ID, sampler)
    s_fullbatch = payload_fullbatch["samples"].double()
    s_minibatch = payload_minibatch["samples"].double()
    s_negmap = payload_negmap["samples"].double()
    label = SAMPLER_STYLE.get(sampler, (None, sampler))[1]
    axes[0].plot(mirror_projection(s_fullbatch, mask, x_ref_pos), lw=1, label=f"{label}-fullbatch")
    #axes[0].plot(mirror_projection(s_minibatch, mask, x_ref_pos), lw=0.8, label=f"{label}-minibatch", ls=":")
    axes[1].plot(mirror_projection(s_negmap, mask, x_ref_pos), lw=1, label=f"{label}-negmap")
    # axes[1].plot(s_fullbatch.norm(dim=1), lw=1)
    # axes[1].plot(s_negmap.norm(dim=1), lw=1)

axes[0].axhline(0, color="0.6", lw=0.8)
axes[1].axhline(0, color="0.6", lw=0.8)
axes[0].set_ylabel("mirror proj")
axes[1].set_ylabel("mirror proj")
axes[1].set_xlabel("draw")
axes[0].legend(fontsize=7, frameon=False)
axes[1].legend(fontsize=7, frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
mask = flip_mask(bm)
x_ref_pos = reference_x_ref(DATASET, SPLIT_ID, bm, data)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True, sharey=True)
for sampler in ['grid_boomerang', 'grid_sticky_boomerang']:
    payload_fullbatch = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
    payload_minibatch = load_run("minibatch", DATASET, SPLIT_ID, sampler)
    payload_negmap = load_run("negmap", DATASET, SPLIT_ID, sampler)
    s_fullbatch = payload_fullbatch["samples"].double()
    s_minibatch = payload_minibatch["samples"].double()
    s_negmap = payload_negmap["samples"].double()
    label = SAMPLER_STYLE.get(sampler, (None, sampler))[1]
    axes[0].plot(mirror_projection(s_fullbatch, mask, x_ref_pos), lw=1, label=f"{label}-fullbatch")
    #axes[0].plot(mirror_projection(s_minibatch, mask, x_ref_pos), lw=0.8, label=f"{label}-minibatch", ls=":")
    axes[1].plot(mirror_projection(s_negmap, mask, x_ref_pos), lw=1, label=f"{label}-negmap")
    # axes[1].plot(s_fullbatch.norm(dim=1), lw=1)
    # axes[1].plot(s_negmap.norm(dim=1), lw=1)

axes[0].axhline(0, color="0.6", lw=0.8)
axes[1].axhline(0, color="0.6", lw=0.8)
axes[0].set_ylabel("mirror proj")
axes[1].set_ylabel("mirror proj")
axes[1].set_xlabel("draw")
axes[0].legend(fontsize=7, frameon=False)
axes[1].legend(fontsize=7, frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
coord = int(mask.nonzero()[1])  # first flipped coordinate

fig, ax = plt.subplots(2, 1, figsize=(8, 2.5), sharex=True, sharey=True)
for sampler in ['grid_zigzag', 'grid_sticky_zigzag']:
    payload_fullbatch = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
    payload_minibatch = load_run("minibatch", DATASET, SPLIT_ID, sampler)
    payload_negmap = load_run("negmap", DATASET, SPLIT_ID, sampler)

    s_fullbatch = payload_fullbatch["samples"].double()
    s_minibatch = payload_minibatch["samples"].double()
    s_negmap = payload_negmap["samples"].double()
    label = SAMPLER_STYLE.get(sampler, (None, sampler))[1]
    ax[0].plot(s_fullbatch[:, coord], lw=1, label=f"{label}-fullbatch")
    ax[1].plot(s_minibatch[:, coord], lw=1, label=f"{label}-negmap")

ax[0].axhline(float(x_ref_pos[coord]), color="0.4", ls="--", lw=0.8)
ax[1].axhline(-float(x_ref_pos[coord]), color="0.4", ls=":", lw=0.8)
ax[0].set_ylabel(f"$\\beta_{{{coord}}}$")
ax[1].set_ylabel(f"$-\\beta_{{{coord}}}$")
ax[0].set_xlabel("draw")
ax[0].legend(fontsize=7, frameon=False)
ax[1].legend(fontsize=7, frameon=False)
plt.show()


In [ ]:
coord = int(mask.nonzero()[1])  # first flipped coordinate

fig, ax = plt.subplots(2, 1, figsize=(8, 2.5), sharex=True, sharey=True)
for sampler in ['grid_boomerang', 'grid_sticky_boomerang']:
    payload_fullbatch = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
    payload_minibatch = load_run("minibatch", DATASET, SPLIT_ID, sampler)
    payload_negmap = load_run("negmap", DATASET, SPLIT_ID, sampler)

    s_fullbatch = payload_fullbatch["samples"].double()
    s_minibatch = payload_minibatch["samples"].double()
    s_negmap = payload_negmap["samples"].double()
    label = SAMPLER_STYLE.get(sampler, (None, sampler))[1]
    ax[0].plot(s_fullbatch[:, coord], lw=1, label=f"{label}-fullbatch")
    ax[1].plot(s_minibatch[:, coord], lw=1, label=f"{label}-negmap")

ax[0].axhline(float(x_ref_pos[coord]), color="0.4", ls="--", lw=0.8)
ax[1].axhline(-float(x_ref_pos[coord]), color="0.4", ls=":", lw=0.8)
ax[0].set_ylabel(f"$\\beta_{{{coord}}}$")
ax[1].set_ylabel(f"$-\\beta_{{{coord}}}$")
ax[0].set_xlabel("draw")
ax[0].legend(fontsize=7, frameon=False)
ax[1].legend(fontsize=7, frameon=False)
plt.show()


---
## Paper results

Posterior predictive mean $\pm 2\sigma$ (epistemic + noise) on the toy dataset below, full training run (no gradient/skeleton budget). NUTS is the reference sampler; minibatch reruns each grid sampler with a subsampled gradient (batch size read from the run) instead of the full training set, everything else unchanged. RMSE/NLL/CRPS are on the held-out test set. `mirror frac>0` is the fraction of posterior draws on the positive side of the tanh sign-flip symmetry: a grid sampler started from the mirror-flipped MAP (`neg-MAP init`) should reach ~0.5 if it mixes across both symmetric modes; a value near 0 means it stayed in the mode it started in. This does not affect predictive accuracy (both modes give the same function), but it means the sampler should not be read as having explored the full symmetric posterior.

In [ ]:
lbbnn = load_run("fullbatch", "sharp", 0, "lbbnn")

In [ ]:
lbbnn

In [ ]:
samplers = GRID_SAMPLERS + ["nuts", "lbbnn"]#, "svi"]
fig, axes = plt.subplots(3, 2, figsize=(7.2, 9.6), sharey=True)
axes = axes.flatten()

for ax, sampler in zip(axes, samplers):
    color, label = SAMPLER_STYLE[sampler]
    ax.plot(x_test[si], y_true[si], color="0.5", lw=1.2, zorder=3)
    ax.scatter(x_train, y_train, marker="x", color="black", s=30, zorder=5)
    for run_key in (["fullbatch"] if sampler in ("nuts", "svi", "lbbnn") else RUN_STYLE):
        payload = load_run(run_key, DATASET, SPLIT_ID, sampler)
        if payload is None:
            continue
        rlabel, ls, lw = RUN_STYLE[run_key]
        mean, epi, total = predictive_summary(payload["samples"], bm, payload, data, x_grid)
        if run_key != "minibatch":
            ax.fill_between(x_grid, mean - 2 * total, mean + 2 * total, color=color, alpha=0.15, linewidth=0)
        ax.plot(x_grid, mean, color=color, ls=ls, lw=lw, zorder=4, label=rlabel)
    ax.set_title(label, fontsize=15)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

#handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.02))
#fig.suptitle(DATASET, fontsize=15)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(f"/Users/augustarnstad/Documents/sticky_samplers/sticky_bnns/results/plots/toy_bnns/{DATASET}.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
data = load_dataset(DATASET)
bm = make_bm(DATASET, data)

PANELS = [
    ["grid_boomerang", "grid_zigzag", "nuts"],
    ["grid_sticky_boomerang", "grid_sticky_zigzag", "lbbnn"],
]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)

for ax, group in zip(axes, PANELS):
    ax.plot(x_test[si], y_true[si], color="0.5", lw=1.2, zorder=3, label="Truth")
    #ax.scatter(x_train, y_train, marker="x", color="black", s=30, zorder=5)
    for sampler in group:
        color, label = SAMPLER_STYLE[sampler]
        payload = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
        if payload is None:
            continue
        _, ls, lw = RUN_STYLE["fullbatch"]
        mean, epi, total = predictive_summary(payload["samples"], bm, payload, data, x_grid)
        ax.fill_between(x_grid, mean - 2 * total, mean + 2 * total, color=color, alpha=0.15, linewidth=0)
        ax.plot(x_grid, mean, color=color, ls=ls, lw=lw, zorder=4, label=label)
    ax.legend(frameon=False, fontsize=10, loc="best")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

fig.tight_layout()
fig.savefig(f"/Users/augustarnstad/Documents/sticky_samplers/sticky_bnns/results/plots/toy_bnns/{DATASET}_pair.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
data = load_dataset(DATASET)
bm = make_bm(DATASET, data)

PANELS = [
    ["grid_boomerang", "grid_zigzag", "nuts"],
    ["grid_sticky_boomerang", "grid_sticky_zigzag", "lbbnn"],
]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)

for ax, group in zip(axes, PANELS):
    for sampler in group:
        color, label = SAMPLER_STYLE[sampler]
        payload = load_run("fullbatch", DATASET, SPLIT_ID, sampler)
        if payload is None:
            continue
        _, ls, lw = RUN_STYLE["fullbatch"]
        mean, epi, total = predictive_summary(payload["samples"], bm, payload, data, x_grid)
        #ax.fill_between(x_grid, mean - 2 * total, mean + 2 * total, color=color, alpha=0.15, linewidth=0)
        ax.plot(x_grid, mean - 2 * total, color=color, alpha=0.3, linewidth=1)
        ax.plot(x_grid, mean + 2 * total, color=color, alpha=0.3, linewidth=1)
        ax.plot(x_grid, mean, color=color, ls=ls, lw=1.8, zorder=4, label=label)
    ax.legend(frameon=False, fontsize=10, loc="best")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

fig.tight_layout()
fig.savefig(f"/Users/augustarnstad/Documents/sticky_samplers/sticky_bnns/results/plots/toy_bnns/{DATASET}_pair_NUTS_truth.png", dpi=150, bbox_inches="tight")
plt.show()


## Mode-crossing check: pairwise weight marginals

For a sampler, picks a few flip-mask coordinates (weights that flip sign between the two tanh-symmetric modes) and scatters `fullbatch` vs `neg-MAP init` draws directly in weight space. `x_ref_pos`/`x_ref_neg` mark the two mode centers. If the two colored clouds overlap, that sampler is genuinely crossing between modes on these coordinates; if they stay in separate corners, it isn't.

In [ ]:
N_DIMS = 3      # 2 or 3
SAMPLER = "grid_boomerang" # "grid_zigzag"
is_boom = "boomerang" in SAMPLER

mask = flip_mask(bm)
x_ref_pos = reference_x_ref(DATASET, SPLIT_ID, bm, data)
x_ref_neg = torch.where(mask, -x_ref_pos, x_ref_pos)

flip_idx = (mask & weight_mask(bm)).nonzero().flatten()
# Rank flip-mask coords by how far apart the two mode centers actually are
# there (sampler-agnostic -- no sticky/frozen assumption).
separation = (x_ref_pos[flip_idx] - x_ref_neg[flip_idx]).abs()
coords = flip_idx[separation.argsort(descending=True)[:N_DIMS]].tolist()

fb_skel = load_run("fullbatch", DATASET, SPLIT_ID, SAMPLER, skeletons=True)
nm_skel = load_run("negmap", DATASET, SPLIT_ID, SAMPLER, skeletons=True)

if USE_SKELETONS and (fb_skel is not None or nm_skel is not None):
    x_ref_arg = x_ref_pos if is_boom else None
    path_fb = reconstruct_path(fb_skel, coords, x_ref=x_ref_arg) if fb_skel is not None else None
    path_nm = reconstruct_path(nm_skel, coords, x_ref=x_ref_arg) if nm_skel is not None else None
    plot_title_suffix = "full trajectory (from skeleton)"
else:
    if USE_SKELETONS:
        print("USE_SKELETONS=True but no skeleton found for this sampler/dataset -- "
              "falling back to resampled draws. Re-run with --save-skeleton to get "
              "the full reconstructed path here instead.")
    else:
        print("USE_SKELETONS=False: showing resampled draws. Skeletons are available "
              "for this run -- set USE_SKELETONS=True above to see the full "
              "reconstructed continuous path instead." if (fb_skel is not None or nm_skel is not None)
              else "USE_SKELETONS=False: showing resampled draws (no skeletons saved for this run).")
    fb = load_run("fullbatch", DATASET, SPLIT_ID, SAMPLER)
    nm = load_run("negmap", DATASET, SPLIT_ID, SAMPLER)
    path_fb = fb["samples"][:20_000, coords].double().numpy() if fb is not None else None
    path_nm = nm["samples"][:20_000, coords].double().numpy() if nm is not None else None
    plot_title_suffix = "resampled draws"


In [ ]:
from matplotlib.legend_handler import HandlerLine2D
import matplotlib.lines as mlines

fig, axes = plt.subplots(N_DIMS, N_DIMS, figsize=(8, 6))
for i in range(N_DIMS):
    for j in range(N_DIMS):
        ax = axes[i, j]
        if i == j:
            if path_fb is not None:
                ax.hist(path_fb[:, i], bins=60, color="green", alpha=0.5, density=True)
            if path_nm is not None:
                ax.hist(path_nm[:, i], bins=60, color="red", alpha=0.5, density=True)
            ax.set_yticks([])
        elif i > j:
            plot_fn = (lambda ax, x, y, **kw: ax.plot(x, y, lw=3.0, **kw)) if USE_SKELETONS else \
                      (lambda ax, x, y, **kw: ax.scatter(x, y, s=3, **kw))
            if path_fb is not None:
                plot_fn(ax, path_fb[:, j], path_fb[:, i], alpha=0.5, color="green", label="MAP")
            if path_nm is not None:
                plot_fn(ax, path_nm[:, j], path_nm[:, i], alpha=0.5, color="red", label="Symmetric MAP")
            ax.scatter([float(x_ref_pos[coords[j]])], [float(x_ref_pos[coords[i]])],
                       marker="x", color="black", s=60, zorder=5)
            ax.scatter([float(x_ref_neg[coords[j]])], [float(x_ref_neg[coords[i]])],
                       marker="x", color="0.4", s=60, zorder=5)
        else:
            ax.set_visible(False)
        if i == N_DIMS - 1 and j==0:
            ax.set_xlabel(f"$\\beta_{1}$", fontsize=15)
        if i == N_DIMS - 1 and j==1:
            ax.set_xlabel(f"$\\beta_{2}$", fontsize=15)
        if i == N_DIMS - 1 and j==2:
            ax.set_xlabel(f"$\\beta_{3}$", fontsize=15)
        if j == 0 and i == 0:
            ax.set_ylabel(f"$\\beta_{1}$", fontsize=15)
        if j == 0 and i == 1:
            ax.set_ylabel(f"$\\beta_{2}$", fontsize=15)
        if j == 0 and i == 2:
            ax.set_ylabel(f"$\\beta_{3}$", fontsize=15)

handles, labels = axes[1, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, fontsize=15, bbox_to_anchor=(0.7, +0.7),
           handlelength=3, markerscale=15)#-0.02))
fig.suptitle(f"Boomerang", fontsize=20)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(f"/Users/augustarnstad/Documents/sticky_samplers/sticky_bnns/results/plots/toy_bnns/{DATASET}_negative_MAP_{SAMPLER}.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
run = load_run("fullbatch", "sharp", 0, "grid_sticky_zigzag")

In [ ]:
frac_zero_per_sample = (run["samples"] == 0).float().mean(dim=1)
mean_sparsity = frac_zero_per_sample.mean().item()
print(mean_sparsity)


In [ ]:
mean_num_zero = (run["samples"] == 0).float().sum(dim=1).mean().item()
print(mean_num_zero)


In [ ]:
# ---------------------------------------------------------------------------
# Mode-crossing check for the STICKY samplers, reconstructed from skeletons.
#
# Two differences from the non-sticky cells above, both required:
#   1. Frozen coordinates must be pinned to exactly 0, not interpolated --
#      mirrors resample_*_path_sticky_torch's `frozen` override
#      (|pos|<zero_tol & |vel|<zero_tol at the LEFT skeleton endpoint).
#      reconstruct_path() does no such thing, so it cannot be used here.
#   2. Each arm's Boomerang orbit is reconstructed around ITS OWN stored
#      x_ref (the neg-MAP arm orbits x_ref_neg, not x_ref_pos).
# ---------------------------------------------------------------------------
import numpy as np
import torch

STICKY_SAMPLER = "grid_sticky_boomerang"   # or "grid_sticky_zigzag"
N_DIMS_S = 3
N_SUB = 20
ZERO_TOL = 1e-12

is_boom_s = "boomerang" in STICKY_SAMPLER


def reconstruct_path_sticky(skel, coords, x_ref=None, n_sub=N_SUB, zero_tol=ZERO_TOL):
    """Dense continuous trajectory on `coords`, with frozen coords pinned to 0.

    x_ref given  -> Boomerang: x(t) = x_ref + (x_k-x_ref)cos(dt) + v_k sin(dt)
    x_ref None   -> ZigZag:    x(t) = x_k + v_k*dt
    A coordinate frozen at the left endpoint of an interval (pos==0 and
    vel==0) stays exactly 0 across that whole interval, matching
    resample_zigzag_path_sticky_torch / resample_boomerang_path_sticky_torch.
    """
    pos = skel["positions"][:, coords].double()
    vel = skel["velocities"][:, coords].double()
    tim = skel["times"].double()
    frac = torch.linspace(0.0, 1.0, n_sub).double()

    frozen = (pos.abs() < zero_tol) & (vel.abs() < zero_tol)   # [n_events, len(coords)]

    out = []
    for k in range(pos.shape[0] - 1):
        dt = frac * (tim[k + 1] - tim[k])
        if x_ref is not None:
            xr = x_ref[coords].double()
            seg = xr + (pos[k] - xr) * torch.cos(dt)[:, None] + vel[k] * torch.sin(dt)[:, None]
        else:
            seg = pos[k] + vel[k] * dt[:, None]
        seg = torch.where(frozen[k].expand_as(seg), torch.zeros_like(seg), seg)
        out.append(seg)
    return torch.cat(out, dim=0).numpy()


# --- load both arms' sticky skeletons -------------------------------------
fb_skel_s = load_run("fullbatch", DATASET, SPLIT_ID, STICKY_SAMPLER, skeletons=True)
nm_skel_s = load_run("negmap",    DATASET, SPLIT_ID, STICKY_SAMPLER, skeletons=True)
if fb_skel_s is None and nm_skel_s is None:
    raise FileNotFoundError(
        f"No sticky skeletons for {DATASET}/{STICKY_SAMPLER}. "
        "Re-run the grid scripts with --save-skeleton.")

# --- mode centers, and the coords where they are furthest apart -----------
mask_s = flip_mask(bm)
x_ref_pos_s = reference_x_ref(DATASET, SPLIT_ID, bm, data)
x_ref_neg_s = torch.where(mask_s, -x_ref_pos_s, x_ref_pos_s)

flip_idx_s = (mask_s & weight_mask(bm)).nonzero().flatten()
separation_s = (x_ref_pos_s[flip_idx_s] - x_ref_neg_s[flip_idx_s]).abs()
coords_s = flip_idx_s[separation_s.argsort(descending=True)[:N_DIMS_S]].tolist()

# Each arm orbits its OWN reference point; prefer the stored one.
def _arm_xref(skel, fallback):
    if not is_boom_s:
        return None
    if skel is not None and skel.get("x_ref") is not None:
        return skel["x_ref"].double()
    return fallback

path_fb_s = (reconstruct_path_sticky(fb_skel_s, coords_s,
                                     x_ref=_arm_xref(fb_skel_s, x_ref_pos_s))
             if fb_skel_s is not None else None)
path_nm_s = (reconstruct_path_sticky(nm_skel_s, coords_s,
                                     x_ref=_arm_xref(nm_skel_s, x_ref_neg_s))
             if nm_skel_s is not None else None)

for nm_, p_ in [("fullbatch", path_fb_s), ("neg-MAP", path_nm_s)]:
    if p_ is not None:
        print(f"{nm_:10s} path {p_.shape}  exactly-zero frac per coord: "
              f"{np.round((p_ == 0.0).mean(axis=0), 3)}")


In [ ]:
TITLE_S = {"grid_sticky_boomerang": "Sticky Boomerang",
           "grid_sticky_zigzag":    "Sticky ZigZag"}[STICKY_SAMPLER]

fig, axes = plt.subplots(N_DIMS_S, N_DIMS_S, figsize=(8, 6))
for i in range(N_DIMS_S):
    for j in range(N_DIMS_S):
        ax = axes[i, j]
        if i == j:
            if path_fb_s is not None:
                ax.hist(path_fb_s[:, i], bins=60, color="green", alpha=0.5, density=True)
            if path_nm_s is not None:
                ax.hist(path_nm_s[:, i], bins=60, color="red", alpha=0.5, density=True)
            ax.set_yticks([])
        elif i > j:
            if path_fb_s is not None:
                ax.plot(path_fb_s[:, j], path_fb_s[:, i], lw=3.0, alpha=0.5,
                        color="green", label="MAP")
            if path_nm_s is not None:
                ax.plot(path_nm_s[:, j], path_nm_s[:, i], lw=3.0, alpha=0.5,
                        color="red", label="Symmetric MAP")
            ax.scatter([float(x_ref_pos_s[coords_s[j]])], [float(x_ref_pos_s[coords_s[i]])],
                       marker="x", color="black", s=60, zorder=5)
            ax.scatter([float(x_ref_neg_s[coords_s[j]])], [float(x_ref_neg_s[coords_s[i]])],
                       marker="x", color="0.4", s=60, zorder=5)
        else:
            ax.set_visible(False)
        if i == N_DIMS_S - 1:
            ax.set_xlabel(rf"$\beta_{{{j+1}}}$", fontsize=15)
        if j == 0:
            ax.set_ylabel(rf"$\beta_{{{i+1}}}$", fontsize=15)

handles, labels = axes[1, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, fontsize=15,
           bbox_to_anchor=(0.7, 0.7), handlelength=3, markerscale=15)
fig.suptitle(TITLE_S, fontsize=20)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(f"results/plots/toy_bnns/{DATASET}_negative_MAP_{STICKY_SAMPLER}.png",
            dpi=150, bbox_inches="tight")
plt.show()


## COVARIANCE MATRIX TEST

In [ ]:
import torch
from utils.toy_results_utils import load_run

DATASET = "multiscale"   # or: hernandez, gap, sharp
SPLIT_ID = 0
ARM = "fullbatch"        # or "negmap" (on disk: negative_map)

zz = load_run(ARM, DATASET, SPLIT_ID, "grid_zigzag",    skeletons=True)
bm = load_run(ARM, DATASET, SPLIT_ID, "grid_boomerang", skeletons=True)

# Flat keys here -- NOT nested under ['payload'] the way the uci_skeletons
# files are. positions/velocities/times sit at the top level.
pos_zz, vel_zz, t_zz = zz["positions"], zz["velocities"], zz["times"]
pos_bm, vel_bm, t_bm = bm["positions"], bm["velocities"], bm["times"]
x_ref = bm["x_ref"]      # Boomerang's reference centre; needed by the quadrature branch

for name, d in [("zigzag", zz), ("boomerang", bm)]:
    print(f"{name:10s} pos={tuple(d['positions'].shape)}  "
          f"T={float(d['times'][-1]):.1f}  bv={d['bound_violations']}  "
          f"grad/skel={d['gradient_evals']/d['positions'].shape[0]:.1f}")


In [ ]:
import numpy as np
import torch
import arviz as az
from sazz.gpu_friendly.utils.resample import (
    resample_zigzag_path_torch, resample_boomerang_path_torch)

N_RESAMPLE = 20_000
BURNIN_FRAC = 0.2

s_zz = resample_zigzag_path_torch(pos_zz, vel_zz, t_zz,
                                  N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC)
s_bm = resample_boomerang_path_torch(pos_bm, vel_bm, t_bm, x_ref,
                                     N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC)

# ArviZ wants (chain, draw, ...). One chain each.
a_zz = np.asarray(s_zz)[None, ...]
a_bm = np.asarray(s_bm)[None, ...]

# Per-coordinate ESS. The bulk variant is the rank-normalised one from
# Vehtari et al. (2021), which is what az.summary reports by default.
ess_zz = az.ess(az.convert_to_dataset(a_zz), method="bulk")["x"].values
ess_bm = az.ess(az.convert_to_dataset(a_bm), method="bulk")["x"].values

print(f"{'sampler':<12}{'grad_evals':>12}{'ESS min':>10}{'ESS med':>10}"
      f"{'ESS/grad (min)':>16}{'ESS/grad (med)':>16}")
print("-" * 76)
for name, e, d in [("ZigZag", ess_zz, zz), ("Boomerang", ess_bm, bm)]:
    g = d["gradient_evals"]
    print(f"{name:<12}{g:>12d}{e.min():>10.1f}{np.median(e):>10.1f}"
          f"{e.min()/g:>16.2e}{np.median(e)/g:>16.2e}")

# Thinning-cap check: if ESS approaches N_RESAMPLE the grid is too coarse
# to resolve the autocorrelation and the numbers understate the samplers.
for name, e in [("ZigZag", ess_zz), ("Boomerang", ess_bm)]:
    frac = float((e > 0.5 * N_RESAMPLE).mean())
    print(f"{name:10s} max ESS {e.max():8.1f} / {N_RESAMPLE}   "
          f"frac coords above half the cap: {frac:.3f}")


In [ ]:
c = int(np.argmin(ess_zz))
print(f"worst zz coord {c}: ESS {ess_zz[c]:.2f}, bm ESS there {ess_bm[c]:.2f}, "
      f"var zz {float(s_zz[:, c].var()):.3e} bm {float(s_bm[:, c].var()):.3e}")
